# 05b — ¿Cuántos clusters (k) usar? Análisis de estabilidad

## ¿Qué hace este notebook?

En `05_demo_segmentation` agrupamos a los usuarios en `k` clusters. Las métricas puntuales (silueta, CH)
crecen de forma monótona con k, así que no fijan un k natural. Aquí decidimos el k por **estabilidad**,
repitiendo el clustering muchas veces sobre trozos distintos de los datos.

## Glosario rápido

- **Silhouette:** mide cómo de bien separados están los clusters (de -1 a 1; más alto = mejor).
- **ARI (Adjusted Rand Index):** mide cuánto se **parecen dos agrupaciones**. Si clusterizamos dos veces
  sobre datos distintos y el ARI es ≈1, el clustering es **estable** a ese k.
- **Submuestreo:** coger al azar el 80 % de los usuarios. Repetimos 15 veces para ver si el resultado
  cambia mucho o se mantiene.

> Este notebook es **de solo lectura**: reconstruye el espacio del autoencoder desde los ficheros que
> guardó el notebook 05, y **no cambia nada** del pipeline.

## 0 · Librerías y rutas

In [ ]:
import pickle
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Anade src/ al path y reutiliza el helper compartido (mismo patron que 01-04)
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))
from tfm.utils import find_project_root  # noqa: E402

PROCESSED_PATH = find_project_root() / "data" / "processed"
np.random.seed(42)
print("Carpeta de datos:", PROCESSED_PATH)

## 1 · Reconstruimos el espacio del autoencoder (X_ae)

Repetimos las mismas transformaciones que el notebook 05 (codificar variables → escalar → pasar por el
autoencoder) usando los ficheros ya guardados, para trabajar exactamente sobre el mismo espacio.

In [ ]:
users = pd.read_csv(PROCESSED_PATH / "users.csv", dtype={"cp_num": str})

d = users.copy()
d["gender_enc"] = (d["gender"] == "H").astype(int)
d["labor_status_enc"] = d["labor_status"].map({"employed": 2, "unemployed": 1, "inactive": 0}).fillna(0).astype(int)
d["civil_status_enc"] = d["civil_status"].map({"casado": 3, "soltero": 0, "divorciado": 1, "viudo": 2}).fillna(0).astype(int)
d["tiene_coche_enc"] = d["tiene_coche"].astype(int)
d["num_room_enc"] = d["num_room"].map({"menos_3_hab": 1, "3_a_6_hab": 2, "7_mas_hab": 3}).fillna(2).astype(int)


def tamano_hogar(s):
    # MISMA codificacion que parse_hogar() del notebook 05 (size_hogar va de 1 a 5 personas),
    # para validar exactamente el mismo espacio que alli.
    if pd.isna(s): return 3
    s = str(s).strip()
    if s.startswith("1"):   return 1
    elif s.startswith("2"): return 2
    elif s.startswith("3"): return 3
    elif s.startswith("4"): return 4
    else:                   return 5


d["size_hogar_enc"] = d["size_hogar"].apply(tamano_hogar)

def edad_a_grupo(edad):
    # Bandas de edad (mismas que el modelo M2): 18-24, 25-34, 35-44, 45-54, 55-64, 65+
    if pd.isna(edad): return -1
    for i, lim in enumerate([25, 35, 45, 55, 65]):
        if edad < lim: return i
    return 5
d["age_cat"] = d["age"].apply(edad_a_grupo)

# La edad va CATEGORIZADA en bandas (age_cat), igual que en 05 y en M2
DEMO_FEATS = ["age_cat", "gender_enc", "labor_status_enc", "civil_status_enc", "tiene_coche_enc",
              "size_hogar_enc", "num_room_enc", "ipa_class", "mun_type", "distance_type"]
X_raw = d[DEMO_FEATS].fillna(-1).values

# Cargamos el escalador y (si se usó) el autoencoder guardados por 05
with open(PROCESSED_PATH / "demo_scaler.pkl", "rb") as f:
    escalador = pickle.load(f)
with open(PROCESSED_PATH / "demo_ae_used.pkl", "rb") as f:
    se_uso_autoencoder = pickle.load(f)

X_escalada = escalador.transform(X_raw)
if se_uso_autoencoder:
    import tensorflow as tf
    encoder = tf.keras.models.load_model(str(PROCESSED_PATH / "demo_encoder.keras"))
    X_ae = encoder.predict(X_escalada, verbose=0)
else:
    X_ae = X_escalada

print("Espacio de clustering X_ae:", X_ae.shape, "| autoencoder usado:", se_uso_autoencoder)

## 2 · Probamos cada k repitiendo el clustering

Para cada `k` de 2 a 12:
1. Repetimos **15 veces**: cogemos el 80 % de usuarios al azar, hacemos KMeans y guardamos su silhouette.
2. Calculamos el **ARI** entre cada par de esas 15 agrupaciones (proyectadas a todos los usuarios).
   ARI alto = el clustering sale parecido cada vez → estable.

In [ ]:
K_RANGE = range(2, 13)
N_REPETICIONES = 15
FRACCION = 0.8

n_usuarios = len(X_ae)
rng = np.random.RandomState(42)
filas = []

for k in K_RANGE:
    silhouettes = []
    agrupaciones = []   # la etiqueta de cluster de TODOS los usuarios, en cada repetición

    for repeticion in range(N_REPETICIONES):
        # 1) cogemos el 80% de los usuarios al azar
        indices = rng.choice(n_usuarios, int(n_usuarios * FRACCION), replace=False)

        # 2) KMeans sobre ese trozo
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        km.fit(X_ae[indices])

        # silhouette de ese trozo
        sil = silhouette_score(X_ae[indices], km.labels_, sample_size=5000, random_state=42)
        silhouettes.append(sil)

        # etiqueta de cluster de TODOS los usuarios (para comparar entre repeticiones)
        agrupaciones.append(km.predict(X_ae))

    # 3) ARI entre cada par de repeticiones
    aris = []
    for i in range(N_REPETICIONES):
        for j in range(i + 1, N_REPETICIONES):
            aris.append(adjusted_rand_score(agrupaciones[i], agrupaciones[j]))

    silhouettes = np.array(silhouettes)
    aris = np.array(aris)
    filas.append({
        "k": k,
        "sil_media": silhouettes.mean(),
        "sil_lo": np.percentile(silhouettes, 2.5),
        "sil_hi": np.percentile(silhouettes, 97.5),
        "ari_media": aris.mean(),
        "ari_std": aris.std(),
    })

resultados = pd.DataFrame(filas)
print(resultados.round(4).to_string(index=False))

In [ ]:
# Gráficos: silhouette (con su intervalo) y ARI por k
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(resultados["k"], resultados["sil_media"], "o-", color="steelblue")
axes[0].fill_between(resultados["k"], resultados["sil_lo"], resultados["sil_hi"], alpha=0.2, color="steelblue")
axes[0].axvline(10, color="crimson", linestyle="-", label="k=10 (adoptado)")
axes[0].axvline(12, color="gray", linestyle=":", label="k=12 (máx silueta)")
axes[0].set_title("Silhouette (media e intervalo)")
axes[0].set_xlabel("k")
axes[0].legend()

axes[1].errorbar(resultados["k"], resultados["ari_media"], yerr=resultados["ari_std"],
                 fmt="o-", color="darkorange", capsize=3)
axes[1].axhline(0.9, color="gray", linestyle="--", linewidth=1, label="ARI=0.90 (estable)")
axes[1].axvline(10, color="crimson", linestyle="-")
axes[1].axvline(12, color="gray", linestyle=":")
axes[1].set_title("Estabilidad (ARI entre repeticiones)")
axes[1].set_xlabel("k")
axes[1].legend()

plt.tight_layout()
plt.show()

k_mejor_sil = int(resultados.loc[resultados["sil_media"].idxmax(), "k"])
k_mejor_ari = int(resultados.loc[resultados["ari_media"].idxmax(), "k"])
ari_en_10 = float(resultados.loc[resultados["k"] == 10, "ari_media"].iloc[0])
print("k con mayor silhouette :", k_mejor_sil)
print("k más estable (ARI)    :", k_mejor_ari)
print("ARI en k=10            :", round(ari_en_10, 4))

## Conclusión

- La **silueta crece de forma monótona** con k (de 0,35 en k=2 a 0,46 en k=12): el espacio demográfico se
  comporta como un **continuo**, sin un número natural de grupos. Por eso su máximo (k=12) coincide con el
  tope del rango de búsqueda y **no es un criterio fiable**.
- El criterio decisivo es la **estabilidad (ARI)**. Hay máximos locales de estabilidad en k=2 (0,999,
  trivial), k=6 (0,957) y **k=10 (0,977)**, este último claramente por encima de sus vecinos (k=9 ≈ 0,88;
  k=11 ≈ 0,94).
- **Se adopta k=10**: es el clustering más cohesionado (silueta ≈ 0,43) que además es **estable** (ARI
  ≈ 0,98) y queda en el **interior** del rango (no en el límite). k=12 tiene métricas algo mejores pero es
  el tope de búsqueda (riesgo de artefacto); k=6 es más parsimonioso pero menos cohesionado.

Este k=10 se propaga al resto del pipeline (propensión M2, arranque en frío de M3 y modelo inverso).